In [1]:
print("hello world")

hello world


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        # Handle percentage values
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        # Handle regular numeric strings
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    # Clean the dataframes
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    # Get all unique dates
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    
    # Assuming constant number of shares (in millions) - need actual data for accurate calculation
    shares_outstanding = pnl_df.loc['Net Profit\xa0+'] / pnl_df.loc['EPS in Rs'] * 1000000
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        # Get values for current period from Balance Sheet
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        # Get values from PNL Statement
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        # Get values from Cash Flow Statement
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        # Calculate book value
        book_value = equity + reserves
        
        # Calculate market cap (approximation using P/E ratio)
        market_cap = net_profit * 20  # Assuming average P/E of 20
        price = market_cap / shares_outstanding[date]
        
        # Calculate metrics
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding[date]) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding[date]) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        # Growth metrics (need previous period)
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        # Current ratio (approximating current assets as Other Assets)
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        # Debt to equity
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        # Free cash flow per share
        fcf = cash_op + cash_inv  # Simplified FCF calculation
        fcf_per_share = fcf / shares_outstanding[date] if shares_outstanding[date] != 0 else None
        
        # Create metrics object
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.0,
    "price_to_book_ratio": 5.06,
    "price_to_sales_ratio": 4.96,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 0.0,
    "earnings_per_share": 19.84
}

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.0,
    "price_to_book_ratio": 4.84,
    "price_to_sales_ratio": 4.6,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 0.0,
    "earnings_per_share": 22.3
}

Period: 2015-03-01
{
    "ticker":

In [4]:
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

In [12]:
print("pnl_df index values:")
print(bs_df.index.tolist())

pnl_df index values:
['Equity Capital', 'Reserves', 'Borrowings\xa0+', 'Other Liabilities\xa0+', 'Total Liabilities', 'Fixed Assets\xa0+', 'CWIP', 'Investments', 'Other Assets\xa0+', 'Total Assets']


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List
import yfinance as yf
from datetime import datetime, timedelta

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Fetch historical price from Yahoo Finance for a specific date
def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')  # Make date timezone-aware (UTC)
    start_date = date - timedelta(days=5)  # Look back 5 days
    end_date = date + timedelta(days=5)    # Look forward 5 days
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        # Find the index of the closest date
        idx = np.abs((hist.index - date).days).argmin()  # Use np.abs on the days array
        return hist['Close'].iloc[idx]  # Return the Close price for that date
    return None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    # Clean the dataframes
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    # Get all unique dates (reporting periods)
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    
    # Get shares outstanding from Yahoo Finance (latest value)
    stock = yf.Ticker(ticker)
    shares_outstanding = stock.info['sharesOutstanding'] / 1000000  # Convert to millions
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        # Get values for current period from Balance Sheet
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        # Get values from PNL Statement
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        # Get values from Cash Flow Statement
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        # Calculate book value
        book_value = equity + reserves
        
        # Get actual market price from Yahoo Finance
        price = get_closest_price(ticker, date)
        if price is not None:
            market_cap = price * shares_outstanding  # Actual market cap
        else:
            market_cap = net_profit * 20  # Fallback approximation
            price = market_cap / shares_outstanding
        
        # Calculate metrics
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        # Growth metrics (need previous period)
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            prev_price = get_closest_price(ticker, prev_date)
            if prev_price is not None:
                market_cap_prev = prev_price * shares_outstanding
            else:
                market_cap_prev = prev_profit * 20  # Fallback approximation
                prev_price = market_cap_prev / shares_outstanding
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        # Current ratio (approximating current assets as Other Assets)
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        # Debt to equity
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        # Free cash flow per share
        fcf = cash_op + cash_inv  # Simplified FCF calculation
        fcf_per_share = fcf / shares_outstanding if shares_outstanding != 0 else None
        
        # Create metrics object
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.25,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.56,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.0,
    "earnings_per_share": 19.84
}

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.26,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.54,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.66,
    "earnings_per_share": 22.3
}

Period: 2015-03-01
{
    "tick

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List
import yfinance as yf
from datetime import datetime, timedelta

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Fetch historical price from Yahoo Finance for a specific date
def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')  # Make date timezone-aware (UTC)
    start_date = date - timedelta(days=5)  # Look back 5 days
    end_date = date + timedelta(days=5)    # Look forward 5 days
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        idx = np.abs((hist.index - date).days).argmin()
        return hist['Close'].iloc[idx]
    return None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    stock = yf.Ticker(ticker)
    shares_outstanding = stock.info['sharesOutstanding'] / 1000000
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        book_value = equity + reserves
        price = get_closest_price(ticker, date)
        if price is not None:
            market_cap = price * shares_outstanding
        else:
            market_cap = net_profit * 20
            price = market_cap / shares_outstanding
        
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            prev_price = get_closest_price(ticker, prev_date)
            if prev_price is not None:
                market_cap_prev = prev_price * shares_outstanding
            else:
                market_cap_prev = prev_profit * 20
                prev_price = market_cap_prev / shares_outstanding
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        fcf = cash_op + cash_inv
        fcf_per_share = fcf / shares_outstanding if shares_outstanding != 0 else None
        
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Function to evaluate metrics against thresholds
def evaluate_metrics(metrics):
    profitability_thresholds = [
        ('return_on_equity', 0.15),  # Strong ROE above 15%
        ('net_margin', 0.20),        # Healthy profit margins
        ('operating_margin', 0.15),  # Strong operating efficiency
    ]
    
    growth_thresholds = [
        ('revenue_growth', 0.10),    # 10% revenue growth
        ('earnings_growth', 0.10),   # 10% earnings growth
        ('book_value_growth', 0.10), # 10% book value growth
    ]
    
    valuation_thresholds = [
        ('price_to_earnings_ratio', 25), # Reasonable P/E ratio
        ('price_to_book_ratio', 3),      # Reasonable P/B ratio
        ('price_to_sales_ratio', 5),     # Reasonable P/S ratio
    ]
    
    # Convert metrics to a dictionary for easier access
    metrics_dict = metrics.dict()
    
    # Profitability evaluation
    profitability_score = 0
    for metric_name, threshold in profitability_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value > threshold:
            profitability_score += 1
    profitability_result = f"Profitability: {profitability_score}/{len(profitability_thresholds)} thresholds met"
    
    # Growth evaluation
    growth_score = 0
    for metric_name, threshold in growth_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value > threshold:
            growth_score += 1
    growth_result = f"Growth: {growth_score}/{len(growth_thresholds)} thresholds met"
    
    # Valuation evaluation
    valuation_score = 0
    for metric_name, threshold in valuation_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value < threshold:  # Note: < for valuation (lower is better)
            valuation_score += 1
    valuation_result = f"Valuation: {valuation_score}/{len(valuation_thresholds)} thresholds met"
    
    # Health score evaluation
    health_score = 0
    if metrics_dict['current_ratio'] and metrics_dict['current_ratio'] > 1.5:
        health_score += 1
    if metrics_dict['debt_to_equity'] and metrics_dict['debt_to_equity'] < 0.5:
        health_score += 1
    if (metrics_dict['free_cash_flow_per_share'] and metrics_dict['earnings_per_share'] and 
        metrics_dict['free_cash_flow_per_share'] > metrics_dict['earnings_per_share'] * 0.8):
        health_score += 1
    health_result = f"Health Score: {health_score}/3"
    
    return {
        "profitability": profitability_result,
        "growth": growth_result,
        "valuation": valuation_result,
        "health": health_result
    }

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results with threshold evaluation
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))
    evaluation = evaluate_metrics(metric)
    print("\nEvaluation:")
    for category, result in evaluation.items():
        print(f"  {result}")


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.25,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.56,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.0,
    "earnings_per_share": 19.84
}

Evaluation:
  Profitability: 3/3 thresholds met
  Growth: 0/3 thresholds met
  Valuation: 3/3 thresholds met
  Health Score: 1/3

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.26,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.54,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,


/var/folders/2z/yqwddyc959v2vj85vf4bgzth0000gn/T/ipykernel_88501/2693647063.py:163: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  metrics_dict = metrics.dict()


NameError: name 'pnldf' is not defined